# Setup & Imports
Import json, time, pandas, matplotlib.pyplot, ResumeRAG from resume_rag, and JobMatcher with SAMPLE_JOB_DESCRIPTIONS from job_matcher.

In [ ]:
import json  # For handling JSON data
import time  # For measuring execution time
import pandas as pd  # For data manipulation and analysis
import matplotlib.pyplot as plt  # For data visualization
from resume_rag import ResumeRAG  # Importing the ResumeRAG class
from job_matcher import JobMatcher, SAMPLE_JOB_DESCRIPTIONS  # Importing JobMatcher and sample job descriptions

# Initialize RAG System & Index Resumes
Instantiate ResumeRAG, call reset(), then process_and_store('resumes') while timing the indexing operation. Print the indexing time.

In [ ]:
# Instantiate the ResumeRAG system
rag = ResumeRAG()

# Reset the system to clear any existing data
rag.reset()

# Measure the time taken to process and store resumes
start = time.time()
rag.process_and_store('resumes')
indexing_time = time.time() - start

# Print the indexing time
print(f'Indexing time: {indexing_time:.2f}s')

# Explore Document Chunks
Access the ChromaDB collection from the RAG instance. Print the total chunk count with collection.count(). Use collection.peek(5) to sample chunks and display each chunk's candidate_name, section, skills metadata, and a 200-char text preview.

In [ ]:
# Access the ChromaDB collection from the RAG instance
collection = rag.collection

# Get the total chunk count
count = collection.count()
print(f'Total chunks in vector DB: {count}')

# Sample and display 5 chunks
sample = collection.peek(5)
for i, (doc, meta) in enumerate(zip(sample['documents'], sample['metadatas'])):
    print(f'\n--- Chunk {i+1} ---')
    print(f'Candidate: {meta["candidate_name"]}')
    print(f'Section: {meta["section"]}')
    print(f'Skills: {meta["skills"]}')
    print(f'Text: {doc[:200]}...')

# Semantic Search Tests
Define a list of 5 test queries (e.g., 'Python developer with machine learning experience', 'Frontend React developer', 'DevOps engineer with Kubernetes', 'Data scientist with SQL and visualization', 'Mobile app developer'). For each query, call rag.query(query, n_results=3) and print the candidate name, similarity score (1-distance), and section for each result.

In [ ]:
# Define a list of test queries
queries = [
    'Python developer with machine learning experience',
    'Frontend React developer',
    'DevOps engineer with Kubernetes',
    'Data scientist with SQL and visualization',
    'Mobile app developer'
]

# Perform semantic search for each query
for query in queries:
    print(f'\nQuery: {query}')
    results = rag.query(query, n_results=3)  # Get top 3 results for the query
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0], results['metadatas'][0], results['distances'][0]
    )):
        # Print candidate name, similarity score, and section
        print(f'  [{i+1}] {meta["candidate_name"]} (similarity: {1-dist:.3f}) - {meta["section"]}')

# Job Matching - All Positions
Instantiate JobMatcher. Loop over all keys in SAMPLE_JOB_DESCRIPTIONS, call matcher.match_job(job_key), store results in all_results dict and latencies in a list. Print job title, latency, and top 5 matches with candidate name, match_score, and matched_skills.

In [ ]:
# Instantiate the JobMatcher
matcher = JobMatcher()

# Initialize dictionaries and lists to store results and latencies
all_results = {}
latencies = []

# Loop through all job descriptions and perform job matching
for job_key in SAMPLE_JOB_DESCRIPTIONS:
    result = matcher.match_job(job_key)  # Match job using the matcher
    all_results[job_key] = result  # Store the result in the dictionary
    latencies.append(result['latency_seconds'])  # Append latency to the list

    # Print job title, latency, and top 5 matches
    print(f"\n{'='*50}")
    print(f"Job: {result['job_title']}")
    print(f"Latency: {result['latency_seconds']}s")
    for i, m in enumerate(result['top_matches'][:5], 1):
        print(f"  [{i}] {m['candidate_name']} - Score: {m['match_score']}/100")
        print(f"      Matched Skills: {', '.join(m['matched_skills'])}")

# Performance Metrics
Print indexing time, average/min/max query latency from the latencies list, total chunks from the collection count, and total resumes loaded via rag.load_resumes('resumes').

In [ ]:
# Print performance metrics
print(f'Indexing Time: {indexing_time:.2f}s')  # Print the time taken for indexing
print(f'Average Query Latency: {sum(latencies)/len(latencies):.3f}s')  # Calculate and print average latency
print(f'Min Latency: {min(latencies):.3f}s')  # Print the minimum latency
print(f'Max Latency: {max(latencies):.3f}s')  # Print the maximum latency
print(f'Total Chunks: {count}')  # Print the total number of chunks in the collection
print(f'Total Resumes: {len(rag.load_resumes("resumes"))}')  # Print the total number of resumes loaded

# Score Distribution Visualization
Create a 2x3 subplot grid (figsize 15x10). For up to 5 jobs in all_results, plot horizontal bar charts of the top 10 candidates' match scores. Set xlim to 0-100, label axes, turn off the 6th subplot. Save as 'score_distribution.png' at 150 dpi and show.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))  # Create a 2x3 subplot grid
axes = axes.flatten()  # Flatten the axes array for easier indexing

# Loop through up to 5 jobs in all_results
for idx, (job_key, result) in enumerate(all_results.items()):
    if idx >= 5:  # Limit to 5 jobs
        break
    # Extract candidate names and match scores for the top 10 matches
    names = [m['candidate_name'].split()[0] for m in result['top_matches'][:10]]
    scores = [m['match_score'] for m in result['top_matches'][:10]]
    
    ax = axes[idx]  # Get the current subplot
    bars = ax.barh(names[::-1], scores[::-1], color='steelblue')  # Plot horizontal bar chart
    ax.set_xlim(0, 100)  # Set x-axis limits
    ax.set_title(result['job_title'], fontsize=10)  # Set subplot title
    ax.set_xlabel('Match Score')  # Label x-axis

# Turn off the 6th subplot
axes[5].axis('off')

plt.tight_layout()  # Adjust layout to prevent overlap
plt.savefig('score_distribution.png', dpi=150)  # Save the figure as a PNG file
plt.show()  # Display the plot
print('Saved: score_distribution.png')  # Print confirmation message

# Latency Comparison
Extract job titles from SAMPLE_JOB_DESCRIPTIONS. Create a bar chart (figsize 10x5) of latencies per job with 'coral' color. Label ylabel as 'Latency (seconds)', add title, rotate x-ticks 15 degrees. Save as 'latency_comparison.png' at 150 dpi and show.

In [ ]:
job_titles = [SAMPLE_JOB_DESCRIPTIONS[k]['title'] for k in all_results.keys()]  # Extract job titles

plt.figure(figsize=(10, 5))  # Set figure size
plt.bar(job_titles, latencies, color='coral')  # Create bar chart with 'coral' color
plt.ylabel('Latency (seconds)')  # Label y-axis
plt.title('Query Latency by Job Description')  # Add title
plt.xticks(rotation=15)  # Rotate x-ticks by 15 degrees
plt.tight_layout()  # Adjust layout to prevent overlap
plt.savefig('latency_comparison.png', dpi=150)  # Save the figure as a PNG file
plt.show()  # Display the plot
print('Saved: latency_comparison.png')  # Print confirmation message

# Detailed Match Analysis
Build a list of row dicts from all_results containing Job title, Candidate name, Score, comma-joined Matched Skills, and Experience years. Create a pandas DataFrame and print it with to_string(index=False).

In [ ]:
# Create a list of dictionaries for each match result
rows = []
for job_key, result in all_results.items():
    for m in result['top_matches']:
        rows.append({
            'Job': result['job_title'],  # Job title
            'Candidate': m['candidate_name'],  # Candidate name
            'Score': m['match_score'],  # Match score
            'Matched Skills': ', '.join(m['matched_skills']),  # Comma-joined matched skills
            'Experience (yrs)': m['experience_years']  # Experience in years
        })

# Create a pandas DataFrame from the rows
df = pd.DataFrame(rows)

# Print the DataFrame without the index
print(df.to_string(index=False))

# Export Results
Write all_results to 'match_results.json' using json.dump with indent=2 and default=str. Export the DataFrame to 'match_results.csv' with index=False. Print confirmation messages for both files.

In [ ]:
with open('match_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)  # Write all_results to JSON file
print('Results saved to match_results.json')  # Confirmation message for JSON file

df.to_csv('match_results.csv', index=False)  # Export DataFrame to CSV file
print('Results saved to match_results.csv')  # Confirmation message for CSV file